# 07 · Solcast vs Reuniwatt: feature comparison and transfer decision
Answers Zaynab's questions before deciding whether we can **train on Solcast and infer with Reuniwatt**.

| Part | Question |
|---|---|
| 1 · Size of differences | Mean, bias, MAE, RMSE and correlation per feature and plant (Reuniwatt satellite vs Solcast, and each vs the plant sensors). Humidity completed. Where the −28 W/m² GHI bias comes from (hour, month, sky type) |
| 2 · Alignment | Timestamps (shift test vs plant sensors), interval start vs end, sunrise/sunset timing on clear days, plant coordinates (Solcast's own zenith vs computed) |
| 3 · Definitions and units | Ranges of cloud opacity, wind, humidity and clearness; cloud opacity as attenuation vs cover; wind vs the plant wind sensor (height and units) |
| 4 · Monthly and daily view | Monthly metrics for GHI, DNI, DHI and cloud; daily plots for clear and cloudy days |
| 5 · Transfer decision | Train on Solcast → predict with Reuniwatt **forecasts**, per month (Jul–Oct 2026) and per feature set |

**Sources:**
- **Reuniwatt satellite** = `ewec_dev_reuniwatt.bronze.satellite_weather_sts` (observed, 2014–2024).
- **Solcast** = Bronze history tables. Solcast's "pv2" table = our **pv1** (small plant), its "pv1" table = our **pv2**.
- **Plant sensors** = Bronze telemetry.

**Outputs (commit & push both):**
- `results/feature_comparison_report.txt`
- `results/feature_plots/*.png`

Part 5 needs the table from notebook 06 (`uc2_analysis_transfer_15min`).

In [ ]:
dbutils.widgets.text("output_schema", "ewec_dev_powerops.scratch_yash", "Output schema")
dbutils.widgets.text("start", "2023-01-01", "Comparison start")
dbutils.widgets.text("end", "2024-12-31", "Comparison end")
dbutils.widgets.text("pv1_lat", "24.42", "pv1 latitude (approx, confirm)")
dbutils.widgets.text("pv1_lon", "55.08", "pv1 longitude (approx, confirm)")
dbutils.widgets.text("pv2_lat", "24.05", "pv2 latitude (approx, confirm)")
dbutils.widgets.text("pv2_lon", "54.40", "pv2 longitude (approx, confirm)")

In [ ]:
%pip install -q lightgbm

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(REPO, "src"))
for m in [m for m in list(sys.modules) if m.startswith("uc2_solar")]:
    del sys.modules[m]
from datetime import datetime, date
from functools import reduce
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from uc2_solar.spark_utils import add_solar, ceil_to, gulf_to_utc
from uc2_solar.metrics import score

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
W = {k: dbutils.widgets.get(k).strip() for k in ["output_schema", "start", "end", "pv1_lat", "pv1_lon", "pv2_lat", "pv2_lon"]}
S = W["output_schema"]
SITES = {"pv1": {"capacity_mw": 935, "lat": float(W["pv1_lat"]), "lon": float(W["pv1_lon"])},
         "pv2": {"capacity_mw": 1584, "lat": float(W["pv2_lat"]), "lon": float(W["pv2_lon"])}}
CAP = {k: v["capacity_mw"] for k, v in SITES.items()}
SOLCAST_TABLE_FOR = {"pv1": "pv2", "pv2": "pv1"}   # Solcast follows Phase 1 naming (swapped)
OUT = "results"
os.makedirs(f"{OUT}/feature_plots", exist_ok=True)
REPORT = [f"# Solcast vs Reuniwatt: feature comparison and transfer decision · run {datetime.utcnow():%Y-%m-%d %H:%M} UTC",
          f"settings: {W}", "Bias / differences are Reuniwatt satellite minus Solcast unless stated. Irradiance in W/m², temperature °C, humidity %, wind m/s."]


def add(title, df, max_rows=400):
    REPORT.append(f"\n## {title}\n" + (df.head(max_rows).round(3).to_string(index=False) if len(df) else "(no rows)"))
    display(df.head(max_rows))


def note(text):
    REPORT.append(text)
    print(text)

## Load and align all sources at 15 min (UTC, period end)
Satellite duplicates per timestamp are averaged. Sensor values are cleaned (impossible values blanked) and converted Gulf → UTC. Sun position and clear-sky GHI are computed for each plant's coordinates (widgets).

In [ ]:
def to15(df, ts, cols):
    return df.withColumn("period_end", ceil_to(ts, 15)).groupBy("site", "period_end").agg(*[F.avg(c).alias(c) for c in cols])

sat = spark.table("ewec_dev_reuniwatt.bronze.satellite_weather_sts")
sat15 = to15(sat.select(F.lower("site").alias("site"), F.col("datetime").alias("ts"),
                        *[F.col(a).cast("double").alias(f"s_{b}") for a, b in [("ghi", "ghi"), ("dni", "dni"), ("dhi", "dhi"), ("bhi", "bhi"),
                          ("temperature", "temp"), ("relative_humidity", "rh"), ("wind_speed", "wind"), ("cloud_opacity", "cloud"), ("clearness_index", "kt")]]),
             "ts", ["s_ghi", "s_dni", "s_dhi", "s_bhi", "s_temp", "s_rh", "s_wind", "s_cloud", "s_kt"])

SOL_MAP = [("ghi", "ghi"), ("dni", "dni"), ("dhi", "dhi"), ("air_temp", "temp"), ("relative_humidity", "rh"),
           ("wind_speed_10m", "wind"), ("cloud_opacity", "cloud"), ("clearsky_ghi", "cs"), ("zenith", "zen")]
sol_parts = []
for site, t in SOLCAST_TABLE_FOR.items():
    raw = spark.table(f"ewec_dev_powerops.bronze.{t}_history_solcast")
    sol_parts.append(raw.select(F.lit(site).alias("site"), F.col("period_end").cast("timestamp").alias("ts"),
                                *[(F.col(a).cast("double") if a in raw.columns else F.lit(None).cast("double")).alias(f"o_{b}") for a, b in SOL_MAP]))
sol15 = to15(reduce(DataFrame.unionByName, sol_parts), "ts", [f"o_{b}" for _, b in SOL_MAP])

PV = spark.table("ewec_dev_powerops.bronze.power_contango_minute_pv").withColumn("ts_gulf", F.col("DateTime").cast("timestamp"))
irr = lambda c: F.when(F.col(c) >= -20, F.greatest(F.col(c).cast("double"), F.lit(0.0)))  # noqa: E731
rng = lambda c, lo, hi: F.when((F.col(c) > lo) & (F.col(c) <= hi), F.col(c).cast("double"))  # noqa: E731
sens = reduce(DataFrame.unionByName, [
    PV.select(F.lit("pv1").alias("site"), gulf_to_utc("ts_gulf").alias("ts"), irr("PV1_GHI").alias("m_ghi"), rng("PV1_Temp", 0, 60).alias("m_temp"),
              rng("PV1_HumidityPer", 0, 100).alias("m_rh"), F.lit(None).cast("double").alias("m_wind")),
    PV.select(F.lit("pv2").alias("site"), gulf_to_utc("ts_gulf").alias("ts"), irr("PV2_GHI").alias("m_ghi"), rng("PV2_Outdoor_Temp", 0, 60).alias("m_temp"),
              rng("PV2_HumidityPer", 0, 100).alias("m_rh"), rng("PV2_WindSpeed", -0.1, 60).alias("m_wind")),
])
sens15 = to15(sens, "ts", ["m_ghi", "m_temp", "m_rh", "m_wind"])

j = (sat15.join(sol15, ["site", "period_end"], "inner").join(sens15, ["site", "period_end"], "left")
     .where(F.to_date("period_end").between(F.lit(W["start"]), F.lit(W["end"]))))
j = add_solar(j, SITES, "period_end", 15)
tbl = f"{S}.uc2_analysis_feature_compare_15min"
j.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl)
df = spark.table(tbl).toPandas()
df["period_end"] = pd.to_datetime(df["period_end"])
df["gulf"] = df["period_end"] + pd.Timedelta(hours=4)
df["gulf_day"] = (df["gulf"] - pd.Timedelta(minutes=1)).dt.date
df["gulf_hour"] = (df["gulf"] - pd.Timedelta(minutes=1)).dt.hour
df["month"] = df["period_end"].dt.to_period("M").astype(str)
day = df[df["clearsky_ghi"] > 50].copy()
note(f"Rows: {len(df):,} (daytime {len(day):,}); period {df.period_end.min()} → {df.period_end.max()}")

# Part 1 · Size of the differences
Irradiance and cloud: **daytime** (computed clear-sky GHI > 50). Temperature, humidity and wind: **all hours**.
Three comparisons per feature where possible: satellite vs Solcast, satellite vs plant sensor, Solcast vs plant sensor.

In [ ]:
def stats(d, a, b):
    x = d[[a, b]].dropna()
    if len(x) < 10:
        return dict(n=len(x), mean_a=np.nan, mean_b=np.nan, bias=np.nan, mae=np.nan, rmse=np.nan, corr=np.nan)
    e = x[a] - x[b]
    return dict(n=len(x), mean_a=x[a].mean(), mean_b=x[b].mean(), bias=e.mean(), mae=e.abs().mean(),
                rmse=float(np.sqrt((e ** 2).mean())), corr=x[a].corr(x[b]))

FEATS = [("ghi", True), ("dni", True), ("dhi", True), ("cloud", True), ("temp", False), ("rh", False), ("wind", False)]
rows = []
for site in ("pv1", "pv2"):
    for f, daytime in FEATS:
        d = (day if daytime else df)
        d = d[d.site == site]
        pairs = [("satellite vs Solcast", f"s_{f}", f"o_{f}")]
        if f"m_{f}" in d.columns:
            pairs += [("satellite vs sensor", f"s_{f}", f"m_{f}"), ("Solcast vs sensor", f"o_{f}", f"m_{f}")]
        for label, a, b in pairs:
            rows.append({"site": site, "feature": f, "comparison": label, "hours": "day" if daytime else "all", **stats(d, a, b)})
p1 = pd.DataFrame(rows)
add("1 · Mean, bias (a − b), MAE, RMSE, correlation per feature and plant (mean_a = first source, mean_b = second)", p1)

**1b · Where does the GHI bias come from?** Satellite − Solcast GHI by Gulf hour, by month and by sky type, plus each source's bias vs the plant sensor (who is off: satellite, Solcast or both?).

In [ ]:
daily = day.groupby(["site", "gulf_day"]).apply(lambda g: pd.Series({
    "sensor_csi": g["m_ghi"].sum() / g["clearsky_ghi"].sum() if g["m_ghi"].notna().mean() > 0.9 else np.nan,
    "n": len(g)})).reset_index()
q_hi, q_lo = daily["sensor_csi"].quantile(0.85), daily["sensor_csi"].quantile(0.10)
daily["sky"] = np.where(daily["sensor_csi"] >= q_hi, "clear", np.where(daily["sensor_csi"] <= q_lo, "cloudy/dusty", "mixed"))
daily.loc[daily["sensor_csi"].isna(), "sky"] = "unknown"
day = day.merge(daily[["site", "gulf_day", "sky", "sensor_csi"]], on=["site", "gulf_day"], how="left")
note(f"Sky types from the plant GHI sensor's daily clear-sky index: clear ≥ {q_hi:.3f} (top 15% of days), cloudy/dusty ≤ {q_lo:.3f} (bottom 10%).")


def bias_table(by):
    out = []
    for key, g in day.groupby(by):
        key = key if isinstance(key, tuple) else (key,)
        out.append({**dict(zip(by, key)), "n": len(g),
                    "sat_minus_solcast": (g.s_ghi - g.o_ghi).mean(), "sat_minus_sensor": (g.s_ghi - g.m_ghi).mean(),
                    "solcast_minus_sensor": (g.o_ghi - g.m_ghi).mean(), "mean_sensor": g.m_ghi.mean()})
    return pd.DataFrame(out)

add("1b · GHI bias by plant (W/m²)", bias_table(["site"]))
add("1b · GHI bias by Gulf hour", bias_table(["site", "gulf_hour"]))
add("1b · GHI bias by month", bias_table(["site", "month"]))
add("1b · GHI bias by sky type", bias_table(["site", "sky"]))

# Part 2 · Alignment: timestamps, intervals, coordinates
**2a · Shift test vs the plant sensor:** correlation of each source's GHI with the sensor GHI when the source is shifted by −30 … +30 min. The best shift tells us how each source is aligned with what the plant measured.

In [ ]:
shift_rows = []
for site in ("pv1", "pv2"):
    d = df[df.site == site].set_index("period_end").sort_index()
    for src in ("s_ghi", "o_ghi"):
        for k in (-30, -15, 0, 15, 30):
            shifted = d[src].copy()
            shifted.index = shifted.index + pd.Timedelta(minutes=k)
            x = pd.concat([shifted.rename("src"), d["m_ghi"].rename("sensor"), d["clearsky_ghi"]], axis=1).dropna()
            x = x[x.clearsky_ghi > 50]
            shift_rows.append({"site": site, "source": "satellite" if src == "s_ghi" else "Solcast", "shift_min": k, "n": len(x),
                               "corr_with_sensor": x["src"].corr(x["sensor"]), "mae": (x["src"] - x["sensor"]).abs().mean()})
add("2a · GHI vs plant sensor at different shifts (best = highest corr / lowest MAE)", pd.DataFrame(shift_rows))

**2b · Sunrise / sunset timing on clear days:** first and last 15-min period with GHI > 50 W/m² (Gulf time), each source vs the computed clear-sky for the plant's coordinates. A constant offset of ~15 min means a start-vs-end labelling difference; a growing offset suggests a coordinate mismatch.

In [ ]:
clear = day[day["sky"] == "clear"]
ramp = []
for (site, gday), g in df[df.gulf_day.isin(clear.gulf_day.unique())].groupby(["site", "gulf_day"]):
    if gday not in set(clear[clear.site == site].gulf_day):
        continue
    rec = {"site": site, "gulf_day": gday}
    for name, col in [("computed", "clearsky_ghi"), ("sensor", "m_ghi"), ("satellite", "s_ghi"), ("solcast", "o_ghi")]:
        on = g[g[col] > 50]
        rec[f"{name}_first"] = on.gulf.min() if len(on) else pd.NaT
        rec[f"{name}_last"] = on.gulf.max() if len(on) else pd.NaT
    ramp.append(rec)
ramp = pd.DataFrame(ramp)
rr = []
for site, g in ramp.groupby("site"):
    for name in ("sensor", "satellite", "solcast"):
        rr.append({"site": site, "source": name, "clear_days": len(g),
                   "median_morning_offset_min": ((g[f"{name}_first"] - g["computed_first"]).dt.total_seconds() / 60).median(),
                   "median_evening_offset_min": ((g[f"{name}_last"] - g["computed_last"]).dt.total_seconds() / 60).median()})
add("2b · clear-day ramp timing vs computed clear-sky (minutes; + = later than computed)", pd.DataFrame(rr))

**2c · Coordinates and timestamp convention from Solcast's own sun position.** Solcast provides `zenith`. We recompute the zenith for a grid of coordinates and three time conventions (period end, middle, start) and find the best match. That gives Solcast's location and timestamp convention, to compare with the plant coordinates in the widgets.

In [ ]:
def zenith_np(ts, lat, lon):
    doy = ts.dt.dayofyear.values
    hr = (ts.dt.hour + ts.dt.minute / 60 + ts.dt.second / 3600).values
    g = 2 * np.pi / 365 * (doy - 1 + (hr - 12) / 24)
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g) + 0.000907 * np.sin(2 * g)
            - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    eqt = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g) - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    ha = np.radians((hr * 60 + eqt + 4 * lon) / 4 - 180)
    la = np.radians(lat)
    cz = np.clip(np.sin(la) * np.sin(decl) + np.cos(la) * np.cos(decl) * np.cos(ha), -1, 1)
    return np.degrees(np.arccos(cz))

grid_rows = []
for site in ("pv1", "pv2"):
    d = df[(df.site == site) & df.o_zen.notna() & (df.o_zen < 85)].sample(n=min(20000, int(((df.site == site) & df.o_zen.notna()).sum())), random_state=1)
    for conv, shift in [("period end", 0), ("period middle", -7.5), ("period start", -15)]:
        t = d["period_end"] + pd.Timedelta(minutes=shift)
        best = None
        for lat in np.arange(23.0, 25.51, 0.25):
            for lon in np.arange(53.0, 56.51, 0.25):
                err = np.abs(zenith_np(t, lat, lon) - d["o_zen"].values).mean()
                if best is None or err < best[2]:
                    best = (lat, lon, err)
        plant = SITES[site]
        err_plant = np.abs(zenith_np(t, plant["lat"], plant["lon"]) - d["o_zen"].values).mean()
        grid_rows.append({"site": site, "solcast_table": SOLCAST_TABLE_FOR[site], "time_convention": conv, "best_lat": best[0], "best_lon": best[1],
                          "mean_abs_zenith_err_deg": best[2], "err_at_widget_coords_deg": err_plant})
add("2c · Solcast zenith: best-matching coordinates per timestamp convention (lowest error = Solcast's convention/location)", pd.DataFrame(grid_rows))

# Part 3 · Definitions and units
**3a · Value ranges.** Two scales (e.g. 0–1 vs 0–100) mean different units or definitions.

In [ ]:
rng_rows = []
for site in ("pv1", "pv2"):
    d = df[df.site == site]
    for col, label in [("s_cloud", "satellite cloud_opacity"), ("o_cloud", "Solcast cloud_opacity"), ("s_kt", "satellite clearness_index"),
                       ("s_wind", "satellite wind_speed"), ("o_wind", "Solcast wind_speed_10m"), ("m_wind", "plant wind sensor"),
                       ("s_rh", "satellite relative_humidity"), ("o_rh", "Solcast relative_humidity"), ("m_rh", "plant humidity sensor")]:
        x = d[col].dropna()
        if len(x):
            rng_rows.append({"site": site, "variable": label, "n": len(x), "min": x.min(), "p01": x.quantile(.01), "p50": x.median(),
                             "p99": x.quantile(.99), "max": x.max(), "share_zero": (x == 0).mean()})
add("3a · value ranges", pd.DataFrame(rng_rows))

**3b · Cloud opacity: attenuation or cover?**
- **Attenuation** (Solcast's documented definition: how much sunlight the cloud removes, 0–100%) should track **1 − GHI ÷ clear-sky**, and be ~0 on clear days.
- **Cover** (fraction of sky covered) tracks less tightly.

We compare each cloud variable with each source's own clear-sky deficit and with the plant sensor's, by sky type.

In [ ]:
day["s_deficit"] = 1 - (day["s_ghi"] / day["clearsky_ghi"]).clip(upper=1.5)
day["o_deficit"] = 1 - (day["o_ghi"] / day["o_cs"]).where(day["o_cs"] > 50).clip(upper=1.5)
day["m_deficit"] = 1 - (day["m_ghi"] / day["clearsky_ghi"]).clip(upper=1.5)
cl = []
for site in ("pv1", "pv2"):
    d = day[day.site == site]
    cl.append({"site": site,
               "corr(sat cloud, sat deficit)": d.s_cloud.corr(d.s_deficit), "corr(sat cloud, 1 - sat clearness)": d.s_cloud.corr(1 - d.s_kt),
               "corr(sat cloud, sensor deficit)": d.s_cloud.corr(d.m_deficit),
               "corr(Solcast cloud, Solcast deficit)": d.o_cloud.corr(d.o_deficit), "corr(Solcast cloud, sensor deficit)": d.o_cloud.corr(d.m_deficit)})
add("3b · cloud opacity vs clear-sky deficit (correlations)", pd.DataFrame(cl))
add("3b · mean cloud opacity by sky type", day.groupby(["site", "sky"])[["s_cloud", "o_cloud", "m_deficit"]].mean().reset_index())

**3c · Wind: height and units.** Both sources vs the plant's wind sensor (pv2 only). A steady ratio around 1.2–1.4 is typical of 10 m vs ~2 m height. A ratio of ~3.6 means km/h vs m/s.

In [ ]:
w = df[(df.site == "pv2")][["s_wind", "o_wind", "m_wind"]].dropna()
add("3c · wind vs plant sensor (pv2, all hours)", pd.DataFrame([
    {"source": "satellite", "mean": w.s_wind.mean(), "sensor_mean": w.m_wind.mean(), "ratio_mean": w.s_wind.mean() / w.m_wind.mean(), "corr": w.s_wind.corr(w.m_wind)},
    {"source": "Solcast (10 m)", "mean": w.o_wind.mean(), "sensor_mean": w.m_wind.mean(), "ratio_mean": w.o_wind.mean() / w.m_wind.mean(), "corr": w.o_wind.corr(w.m_wind)},
]))

# Part 4 · Monthly breakdown and daily plots

In [ ]:
mrows = []
for (site, month), g in day.groupby(["site", "month"]):
    for f in ("ghi", "dni", "dhi", "cloud"):
        mrows.append({"site": site, "month": month, "feature": f, **stats(g, f"s_{f}", f"o_{f}")})
add("4 · monthly satellite vs Solcast (daytime): mean_a = satellite, mean_b = Solcast", pd.DataFrame(mrows).sort_values(["feature", "site", "month"]))

**Daily plots:** for each plant, the 3 clearest and 3 cloudiest/dustiest days (by the plant sensor), each showing GHI (with sensor and computed clear-sky), DNI, DHI and cloud opacity. Saved to `results/feature_plots/`.

In [ ]:
for site in ("pv1", "pv2"):
    ds = daily[(daily.site == site) & daily.sensor_csi.notna() & (daily.n >= 40)].sort_values("sensor_csi")
    picks = [("clear", d) for d in ds.tail(3).gulf_day] + [("cloudy/dusty", d) for d in ds.head(3).gulf_day]
    fig, axes = plt.subplots(len(picks), 4, figsize=(20, 3.2 * len(picks)), squeeze=False)
    for i, (kind, gd) in enumerate(picks):
        g = df[(df.site == site) & (df.gulf_day == gd)].sort_values("gulf")
        for k, (f, ttl) in enumerate([("ghi", "GHI"), ("dni", "DNI"), ("dhi", "DHI"), ("cloud", "cloud opacity")]):
            ax = axes[i][k]
            ax.plot(g.gulf, g[f"s_{f}"], label="Reuniwatt satellite")
            ax.plot(g.gulf, g[f"o_{f}"], label="Solcast")
            if f == "ghi":
                ax.plot(g.gulf, g["m_ghi"], label="plant sensor", color="black", lw=1)
                ax.plot(g.gulf, g["clearsky_ghi"], label="computed clear-sky", color="grey", ls="--", lw=1)
            ax.set_title(f"{site} · {gd} ({kind}) · {ttl}", fontsize=9)
            ax.tick_params(labelsize=7)
            if i == 0:
                ax.legend(fontsize=7)
    fig.tight_layout()
    path = f"{OUT}/feature_plots/{site}_daily_comparison.png"
    fig.savefig(path, dpi=90)
    display(fig)
    plt.close(fig)
    note(f"Saved plot {path} (days: {', '.join(f'{d} {k}' for k, d in picks)})")

# Part 5 · Transfer decision: train on Solcast, predict with Reuniwatt **forecasts**
Uses `uc2_analysis_transfer_15min` (from notebook 06): Solcast and Reuniwatt day-ahead (12–24 h) features with actual output, daytime 06:00–18:00 Gulf.

- **Solcast model:** trained on Solcast, 1 Jan 2024 – 18 Nov 2025, fixed.
- **Reuniwatt model:** trained on Reuniwatt months *before* each test month (no Reuniwatt model for July).

**Feature sets:**
- **full:** GHI, DNI, DHI, temperature + computed.
- **ghi_only:** drops DNI and DHI, which disagree between providers.
- **computed_only:** sun position and time only. A no-weather reference.

In [ ]:
T = spark.table(f"{S}.uc2_analysis_transfer_15min").toPandas()
T["period_end"] = pd.to_datetime(T["period_end"])
T["day"] = T["period_end"].dt.date
T["month"] = T["period_end"].dt.to_period("M").astype(str)
T["cap"] = T["site"].map(CAP)
T["y"] = T["actual_mw"] / T["cap"]
T = T[(T.gulf_hour > 6) & (T.gulf_hour <= 18) & T.y.notna() & (T.n_min >= 8)]
COMP = ["sun_elevation", "cos_zenith", "clearsky_ghi", "hour_sin", "hour_cos", "doy_sin", "doy_cos"]
FSETS = {"full": ["ghi", "dni", "dhi", "temp", "csi"] + COMP, "ghi_only": ["ghi", "temp", "csi"] + COMP, "computed_only": COMP}


def fit(d, cols):
    import lightgbm as lgb
    m = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31, min_child_samples=30, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, verbose=-1)
    m.fit(d[cols].astype("float32"), d["y"])
    return m

sc_tr = T[(T.source == "solcast") & (T.day >= date(2024, 1, 1)) & (T.day <= date(2025, 11, 18))]
ru = T[T.source == "reuniwatt"]
res = []
for site in ("pv1", "pv2"):
    s_models = {k: fit(sc_tr[sc_tr.site == site], c) for k, c in FSETS.items()}
    for month in sorted(ru[ru.site == site].month.unique()):
        te = ru[(ru.site == site) & (ru.month == month)].copy()
        if len(te) < 200:
            continue
        te["cap_mw"] = CAP[site]
        res.append({"site": site, "test_month": month, "model": "Reuniwatt power forecast", **score(te.assign(p=te["power"]), "p", "actual_mw", "cap_mw", 150)})
        for k, c in FSETS.items():
            te["p"] = np.clip(s_models[k].predict(te[c].astype("float32")), 0, 1.05) * CAP[site]
            res.append({"site": site, "test_month": month, "model": f"Solcast-trained · {k}", **score(te, "p", "actual_mw", "cap_mw", 150)})
        r_tr = ru[(ru.site == site) & (ru.month < month)]
        if len(r_tr) > 1500:
            for k in ("full", "ghi_only"):
                te["p"] = np.clip(fit(r_tr, FSETS[k]).predict(te[FSETS[k]].astype("float32")), 0, 1.05) * CAP[site]
                res.append({"site": site, "test_month": month, "model": f"Reuniwatt-trained · {k} ({len(r_tr)} rows)", **score(te, "p", "actual_mw", "cap_mw", 150)})
res = pd.DataFrame(res)
add("5 · transfer test by month (MAPE where actual > 150 MW; nMAE % of capacity)", res.sort_values(["site", "test_month", "mape_pct"]))
summ = res.assign(model_family=res.model.str.replace(r" \(\d+ rows\)", "", regex=True)).groupby(["site", "model_family"])[["mape_pct", "nmae_pct"]].mean().reset_index()
add("5 · average over test months", summ.sort_values(["site", "mape_pct"]))

## Conclusion guide
| Evidence | Supports "train on Solcast, infer with Reuniwatt" if… |
|---|---|
| Part 1 | Biases are small relative to MAE vs the sensors, and GHI and temperature agree |
| Part 2 | Both sources are aligned with the sensor (best shift 0), with similar clear-day ramp timing |
| Part 3 | Cloud and wind definitions differ → exclude them from the shared feature set |
| Part 5 | The Solcast-trained model (full or ghi_only) beats the Reuniwatt forecast **in every month**, and is close to the Reuniwatt-trained model |

In [ ]:
text = "\n".join(REPORT)
with open(f"{OUT}/feature_comparison_report.txt", "w") as fh:
    fh.write(text)
print(f"Saved {os.path.abspath(OUT)}/feature_comparison_report.txt and feature_plots/*.png — commit & push both.")